# Data split notes
- Use the 500 official test pairs as test set
- Select the trainval sets of group44000, group50600, group77000 as validation set
- Use the rest as training set

# Intersection over union (IoU) notes
Use 0.5 as IoU threshold

# Import libraries

In [1]:
import shutil
import os
from pathlib import Path

import pandas as pd

# Convert DeepPCB dataset to YOLO format
You need to download the dataset yourself

In [2]:
# DeepPCB image dimensions
img_width = 640
img_height = 640

In [3]:
# Create PCBData_yolo directories if not already existing
for folder in ["images", "labels"]:
    for set in ["train", "val", "test"]:
        Path(f"PCBData_yolo/{folder}/{set}").mkdir(parents=True, exist_ok=True)

In [4]:
# Convert annotations to YOLO format
def convert_annotations_to_yolo(annotations: pd.DataFrame):
    x_center = ((annotations['x1'] + annotations['x2']) / 2 / img_width).rename("x_center")
    y_center = ((annotations['y1'] + annotations['y2']) / 2 / img_height).rename("y_center")
    width = ((annotations['x2'] - annotations['x1']) / img_width).rename("width")
    height = ((annotations['y2'] - annotations['y1']) / img_height).rename("height")

    return pd.concat([annotations['type'], x_center, y_center, width, height], axis=1)

In [ ]:
# Copy images and annotations from the test set
test_set_paths = pd.read_csv("PCBData/test.txt", sep=" ", header=None, names=["image", "label"])

for image_path in test_set_paths['image']:
    # Convert image path in test set file to path of tested image
    test_image_filename = f"{image_path.split(".")[0]}_test"  # Filename without extension
    test_image_path = ".".join([test_image_filename, image_path.split(".")[1]])

    shutil.copy2(os.path.join("PCBData", test_image_path), "PCBData_yolo/images/test")

for label_path in test_set_paths['label']:
    # Convert labels to YOLO format
    annotations = pd.read_csv(
        os.path.join("PCBData", label_path), 
        sep=" ", 
        header=None, 
        names=["x1", "y1", "x2", "y2", "type"]
    )

    # Export the converted labels file
    label_filename = os.path.split(label_path)[-1]
    annotations_yolo = convert_annotations_to_yolo(annotations)
    annotations_yolo.to_csv(
        os.path.join("PCBData_yolo/labels/test/", label_filename), 
        sep=" ", 
        header=False,
        index=False
    )